In [1]:
import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

import tensorflow as tf

import gymnasium as gym
import pickle

from VALIDATION.KeyRef_2.KeyRef_2_env2 import Luo_DDQN_env
from stable_baselines3.common.callbacks   import BaseCallback
from stable_baselines3.common.env_checker import check_env

import datetime
import pandas as pd
from stable_baselines3.common.callbacks import BaseCallback

K = 30
planning_horizon        = 480*60
ReworkProbability       = 0.03
critical_machines       = {5, 6, 7, 8, 9, 10, 11, 12, 13, 21, 22, 26, 27}


purpose                 = "_loose_duedate_JA_MB"
directory               = 'DATA/LOOSE_DUEDATE'

tight_duedate_setting   = True if "tight_duedate" in purpose else False
JA_only_setting         = True if "JA_only" in purpose else False

WeibullDistribution     = pd.read_excel('DATA/DataMaster.xlsx', sheet_name='Distribution')

with open('DATA/ComponentMaster.pkl', 'rb') as f:
    master = pickle.load(f)
with open(f'{directory}_VALIDATION/pickle_valid_scenarios_480.pkl', 'rb') as f:
    valid_scenarios = pickle.load(f)

env = Luo_DDQN_env(K, planning_horizon, ReworkProbability, valid_scenarios, WeibullDistribution, critical_machines,
			  	 master, tight_duedate_setting, JA_only_setting, directory)

# check_env(env)
# obs = env.reset(seed=42,
#                 test=True, 
#                 datatest="valid2", 
#                 scenariotest="A")
            
# print("Observation:", obs)

# episodes = 1
# for episode in range(episodes):
# 	done = False
# 	obs = env.reset(test=True, 
#                     datatest="valid2", 
#                     scenariotest="A")
# 	while done == False: #not done:
# 		random_action = env.action_space.sample()
# 		obs, reward, done, truncated, info = env.step(random_action)

In [2]:
action_list = ["CDR1", "CDR2", "CDR3", "CDR4", "CDR5", "CDR6"]
                          
# Create directories for models and logs
current_time = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
models_dir = f"models/LuoDDQN{purpose}-{current_time}"
logdir = f"logs/LuoDDQN{purpose}-{current_time}"
log_training_txt_dir = "keyref2_log_training_txt"
log_training_excel_dir = "keyref2_log_training_excel"

if not os.path.exists(models_dir):
    os.makedirs(models_dir)
if not os.path.exists(logdir):
    os.makedirs(logdir)
if not os.path.exists(log_training_txt_dir):
    os.makedirs(log_training_txt_dir)
if not os.path.exists(log_training_excel_dir):
    os.makedirs(log_training_excel_dir)

# Generate unique file names based on current time
log_file          = os.path.join(log_training_txt_dir,   f"training_keyref2.txt")
excel_file        = os.path.join(log_training_excel_dir, f"training_keyref2.xlsx")
action_count_file = os.path.join(log_training_txt_dir,   f"action_count_keyref2.txt")
action_excel_file = os.path.join(log_training_excel_dir, f"action_count_keyref2.xlsx")

# Define the custom callback -------------------------------------------------------------
class CustomCallback(BaseCallback):
    def __init__(self, log_dir, excel_file, txt_file, action_count_file, action_excel_file, verbose=0):
        super(CustomCallback, self).__init__(verbose)
        self.log_dir = log_dir
        self.excel_file = excel_file
        self.txt_file = txt_file
        self.action_count_file = action_count_file
        self.action_excel_file = action_excel_file
        self.logs = []
        self.episode_rewards = []
        self.action_counts = {}
        self.episode_start = True

    def _on_training_start(self) -> None:
        # Initialize action counts
        self.action_counts = {action: 0 for action in action_list}

    def _on_step(self) -> bool:
        if self.episode_start:
            self.episode_rewards.append(0)
            self.episode_start = False

        # Record reward for the current step
        reward = self.locals['rewards'][0]
        self.episode_rewards[-1] += reward

        # Increment action count
        action = self.locals.get('actions', None)
        if action is not None:
            action_name = action_list[action[0]]
            self.action_counts[action_name] += 1
            
        return True

    def _on_rollout_end(self) -> None:
        # Called at the end of each episode
        sum_reward   = self.episode_rewards[-1] if self.episode_rewards else 0
        
        self.logger.record('train/episode_reward',   sum_reward)
        self.logs.append({
            'sum_reward': sum_reward,
        })

        self.episode_start = True

    
    def _on_training_end(self) -> None:
        # Save logs to Excel
        df = pd.DataFrame(self.logs)
        df.to_excel(self.excel_file, index=False)

        action_df = pd.DataFrame(list(self.action_counts.items()), columns=['Action', 'Count'])
        action_df.to_excel(self.action_excel_file, index=False)

        # Save logs to text file
        with open(self.txt_file, 'w') as f:
            f.write(df.to_string(index=False))
        with open(self.action_count_file, 'w') as f:
            f.write(action_df.to_string(index=False))

# Create the callback
callback = CustomCallback(log_dir=logdir, 
                          excel_file=excel_file,
                          txt_file=log_file,
                          action_count_file=action_count_file,
                          action_excel_file=action_excel_file,
                          verbose=1)

import torch
import torch.nn as nn
import torch.nn.functional as F
from gymnasium import spaces
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3 import DQN

class CustomFeaturesExtractor(BaseFeaturesExtractor):
    """
    Custom feature extractor for DQN.
    
    :param observation_space: (spaces.Box)
    """
    def __init__(self, observation_space: spaces.Box):
        super(CustomFeaturesExtractor, self).__init__(observation_space, features_dim=6)  # Output features_dim matches last layer's output
        n_input_nodes = observation_space.shape[0]
        self.fc1 = nn.Linear(n_input_nodes, 30)
        self.fc2 = nn.Linear(30, 30)
        self.fc3 = nn.Linear(30, 30)
        self.fc4 = nn.Linear(30, 30)
        self.fc5 = nn.Linear(30, 30)
        self.fc6 = nn.Linear(30, 6)  # Output layer with 6 nodes

    def forward(self, observations: torch.Tensor) -> torch.Tensor:
        x = F.tanh(self.fc1(observations))
        x = F.tanh(self.fc2(x))
        x = F.tanh(self.fc3(x))
        x = F.tanh(self.fc4(x))
        x = F.tanh(self.fc5(x))
        x = self.fc6(x)  # Output layer
        return x

# Custom DQN class to implement soft target update
class CustomDQN(DQN):
    def __init__(self, *args, tau=0.01, **kwargs):
        super(CustomDQN, self).__init__(*args, **kwargs)
        self.tau = tau

    def train(self, gradient_steps, batch_size=100):
        # Train for gradient_steps
        for gradient_step in range(gradient_steps):
            # Sample replay buffer
            replay_data = self.replay_buffer.sample(batch_size, env=self._vec_normalize_env)
            
            # Mix online and target networks
            target_q_values = self.q_net_target(replay_data.next_observations)
            next_q_values, _ = target_q_values.max(dim=1)
            next_q_values = next_q_values.reshape(-1, 1)

            # Compute the target for the Q function
            target_q = replay_data.rewards + (1 - replay_data.dones) * self.gamma * next_q_values

            # Get current Q estimates
            current_q = self.q_net(replay_data.observations).gather(1, replay_data.actions.long())

            # Compute Huber loss (less sensitive to outliers)
            loss = F.smooth_l1_loss(current_q, target_q)

            # Optimize the model
            self.policy.optimizer.zero_grad()
            loss.backward()
            self.policy.optimizer.step()

            # Soft update of target network
            with torch.no_grad():
                for target_param, param in zip(self.q_net_target.parameters(), self.q_net.parameters()):
                    target_param.data.copy_(self.tau * param.data + (1.0 - self.tau) * target_param.data)

            self.logger.record('train/loss', loss.item())

# Define policy_kwargs for DQN model
policy_kwargs = dict(
    features_extractor_class=CustomFeaturesExtractor,
)

model_path = os.path.join(models_dir, "CustomDQN_.zip")
# Initialize CustomDQN using the custom model
model = CustomDQN(
    'MlpPolicy',                    # Use a Multi-layer Perceptron (MLP) policy
    env,                            # Your RL environment
    policy_kwargs=policy_kwargs,
    buffer_size=2000,               # Replay buffer size N
    batch_size=32,                  # Batch size
    gamma=0.9,                      # Discount factor
    tau=0.01,                       # Soft target update strategy
    exploration_initial_eps=0.5,    # Initial epsilon
    exploration_final_eps=0.1,      # Final epsilon
    exploration_fraction=0.5,
    verbose=1,
    tensorboard_log=logdir,
    train_freq=(30,"step"),
    learning_starts= 2000,
    learning_rate= 1e-4
)
model.learn(total_timesteps=40000, 
            tb_log_name="Luo DDQN",
            log_interval=1,
            reset_num_timesteps=True,
            callback=callback)
model.save(model_path)



Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
JA_event [(4, 4, 'urgent'), (117, 7, 'urgent'), (123, 7, 'urgent'), (139, 3, 'urgent'), (149, 2, 'urgent'), (161, 3, 'urgent'), (174, 4, 'urgent'), (203, 7, 'urgent'), (326, 3, 'urgent')]
MB_event [[], [], [], [], [], [], [], [], [(18120, 7080, 'critical')], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], []]
Logging to logs/LuoDDQN_loose_duedate_JA_MB-2025-01-12_15-13-47\Luo DDQN_1
====== Done ======
JA_event [(30, 57600, 'loose'), (33, 57600, 'loose'), (104, 57600, 'loose'), (137, 57600, 'loose'), (226, 57600, 'loose'), (274, 57600, 'loose'), (313, 57600, 'loose'), (339, 57600, 'loose'), (410, 57600, 'loose'), (417, 57600, 'loose'), (626, 57600, 'loose'), (783, 57600, 'loose'), (797, 57600, 'loose'), (829, 57600, 'loose'), (855, 57600, 'loose'), (859, 57600, 'loose'), (865, 57600, 'loose'), (916, 57600, 'loose'), (992, 57600, 'loose'), (1018, 57600, 'loose'), 

c:\Users\dvtruc\.conda\envs\dunnbebes\lib\site-packages\numpy\core\fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\dvtruc\.conda\envs\dunnbebes\lib\site-packages\numpy\core\_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


====== Done ======
JA_event [(74, 28800, 'normal'), (102, 28800, 'normal'), (106, 28800, 'normal'), (127, 28800, 'normal'), (131, 28800, 'normal'), (136, 28800, 'normal'), (270, 28800, 'normal'), (282, 28800, 'normal'), (369, 28800, 'normal'), (455, 28800, 'normal'), (462, 28800, 'normal'), (466, 28800, 'normal'), (570, 28800, 'normal'), (587, 28800, 'normal'), (605, 28800, 'normal'), (621, 28800, 'normal'), (740, 28800, 'normal'), (743, 28800, 'normal'), (796, 28800, 'normal'), (832, 28800, 'normal'), (842, 28800, 'normal'), (850, 28800, 'normal')]
MB_event [[], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], []]
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 2.11e+03 |
|    ep_rew_mean      | -871     |
|    exploration_rate | 0.121    |
| time/               |          |
|    episodes         | 9        |
|    fps              | 26       |
|    time_elapsed     | 712      |

In [3]:
import numpy as np
import torch
from stable_baselines3 import DQN

# model_path = "models/LuoDDQN_loose_duedate_JA_MB-2025-01-12_01-21-19/CustomDQN_"

model = DQN.load(model_path, env=env)
def softmax_action_selection(model, obs, mu):
    obs_tensor      = torch.tensor(obs, dtype=torch.float32).unsqueeze(0).to(model.device)
    q_values        = model.q_net(obs_tensor).detach().cpu().numpy().flatten()
    exp_q_values    = np.exp(mu * q_values)
    probabilities   = exp_q_values / np.sum(exp_q_values)
    action          = np.random.choice(len(q_values), p=probabilities)
    return action

results = []
method = 'keyref2'
InstanceList = [f'valid{i+1}' for i in range(2)]
ScenarioList = ['A', 'B', 'C']

mu = 1.6 
for time_run in range(1):
    print('time:', time_run)
    for instance_id in InstanceList:
        print("-----------", instance_id)
        for scenario_id in ScenarioList:
            print("-----", scenario_id)
            # Reset the environment with the new dataset
            obs, info = env.reset(test=True, 
                    datatest=instance_id, 
                    scenariotest=scenario_id)
            
            done = False
            
            while not done:
                action = softmax_action_selection(model, obs, mu)
                obs, reward, done, truncated, info = env.step(action)
            
            tardiness = env.calc_tardiness()
            print(tardiness)
            results.append({
                            'TimeRun'   : time_run,
                            'Method'    : method,
                            'InstanceID': instance_id,
                            'ScenarioID': scenario_id,
                            'Tardiness' : tardiness
                            })



Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
time: 0
----------- valid1
----- A
JA_event [(2, 5, 'urgent'), (23, 9, 'urgent'), (27, 6, 'urgent'), (71, 6, 'urgent'), (98, 9, 'urgent'), (214, 4, 'urgent'), (354, 7, 'urgent'), (403, 10, 'urgent'), (410, 5, 'urgent'), (435, 7, 'urgent'), (473, 9, 'urgent'), (649, 5, 'urgent')]
MB_event [[], [], [], [], [], [], [(3600.0, 2100.0, 'critical')], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], []]
having MB in random_events 6
[[2, 432]]
[[2, 432]]
----------- Random MB at time 40710 on machine [6] at Ope [2, 432]
====== Done ======
3175190.0
----- B
JA_event [(4, 28800, 'normal'), (62, 28800, 'normal'), (224, 28800, 'normal'), (249, 28800, 'normal'), (291, 28800, 'normal'), (302, 28800, 'normal'), (314, 28800, 'normal'), (373, 28800, 'normal'), (471, 28800, 'normal'), (477, 28800, 'normal'), (489, 28800, 'normal'), (495, 28800, 'normal'), (578, 28800, 'normal'), (611, 28800

c:\Users\dvtruc\.conda\envs\dunnbebes\lib\site-packages\numpy\core\fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\dvtruc\.conda\envs\dunnbebes\lib\site-packages\numpy\core\_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


====== Done ======
0.0
----------- valid2
----- A
JA_event [(31, 3, 'urgent'), (52, 8, 'urgent'), (90, 9, 'urgent'), (103, 4, 'urgent'), (222, 9, 'urgent'), (235, 4, 'urgent'), (268, 2, 'urgent'), (289, 10, 'urgent'), (311, 10, 'urgent'), (340, 9, 'urgent'), (347, 6, 'urgent'), (379, 9, 'urgent'), (384, 10, 'urgent')]
MB_event [[], [], [], [], [], [], [], [], [], [(33660.0, 60.0, 'critical')], [(18720.0, 1680.0, 'critical')], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], []]
====== Done ======
189770.0
----- B
JA_event [(21, 28800, 'normal'), (26, 28800, 'normal'), (118, 28800, 'normal'), (135, 28800, 'normal'), (170, 28800, 'normal'), (205, 28800, 'normal'), (208, 28800, 'normal'), (222, 28800, 'normal'), (340, 28800, 'normal'), (365, 28800, 'normal')]
MB_event [[], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], []]
====== Done ======
417890.0
----- C
JA_event [(32, 57600, 'loose'), (50, 57600, 

In [4]:
df = pd.DataFrame(results)
file_name = f"VALIDATION/LuoDDQN_{purpose}_{current_time}_FULL.xlsx"
df.to_excel(file_name, index=False)